# 02 — Context Manager：管理对话历史

上一章写好了 LLM 客户端，能发消息、拿回答。

但现在还有个问题没解决——**对话历史**。

Agent 在运行过程中，对话会越来越长：用户的问题、AI 的回答、工具调用的结果……这些全都要放在 `messages` 列表里发给 LLM。

如果直接用一个裸列表，你很快会遇到这些麻烦：
- 不知道现在用了多少 token，快超限了吗？
- 系统 prompt 怎么管理？每次手动加？
- 工具调用的结果消息格式特殊，怎么加进去？

这章造一个 **Context Manager**，专门管这件事。

---

## 学完这章你能做到

- 用 `MessageItem` 统一表示每条消息，带 token 计数
- 用 `ContextManager` 管理对话历史，自动追踪 token 用量
- 实现 token 计数：tiktoken 精确计数 + 降级方案
- 构造系统 prompt，定义 Agent 的角色和行为规则

## 1. Token 是什么，为什么要数它

LLM 不是按字数收费，也不是按字符数——是按 **token** 收费/计算。

Token 是文本被模型处理前的分词单位，大概的换算关系：
- 英文：1 token ≈ 0.75 个单词（`hello` = 1 token，`unbelievable` = 3 tokens）
- 中文：1 token ≈ 1-2 个汉字（每个汉字通常是 1 个 token）
- 代码：变量名、符号各占 token，所以代码比普通文字贵

**为什么要数 token？**

每个模型有**上下文窗口限制**——gpt-oss:120b 大概是 128k token。超过这个限制，请求会失败。

```
上下文窗口 (128k tokens)
┌─────────────────────────────────────────┐
│ 系统 prompt (约 500 tokens)              │
│ 对话历史 (随对话增长)                     │
│   ├─ 用户消息 1                          │
│   ├─ AI 回答 1                           │
│   ├─ 工具调用结果 (可能很大)              │
│   └─ ...                                │
│ 当前请求                                 │
└─────────────────────────────────────────┘
                                 ↑
               超过这里就报错，要压缩历史（第 08 章）
```

Context Manager 要时刻知道现在用了多少 token，接近上限时能提前处理。

## 2. Token 计数实现

In [ ]:
# !pip install tiktoken openai

In [ ]:
# tiktoken 是 OpenAI 开源的 tokenizer 库
# 可以在不调用 API 的情况下，在本地计算文本的 token 数

try:
    import tiktoken
    # cl100k_base 是 GPT-4 / GPT-3.5-turbo 使用的编码方式
    # 对 Ollama 本地模型来说是近似值，但足够用
    _encoder = tiktoken.get_encoding("cl100k_base")
    
    def count_tokens(text: str) -> int:
        return len(_encoder.encode(text))
    
    print("✓ 使用 tiktoken 精确计数")

except ImportError:
    # tiktoken 装不上（比如某些系统环境），降级到粗略估算
    def count_tokens(text: str) -> int:
        return len(text) // 4  # 粗略：4 个字符 ≈ 1 token
    
    print("⚠ tiktoken 不可用，使用 len/4 粗略估算")


# 验证一下
examples = [
    "Hello, world!",
    "什么是递归？",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

for text in examples:
    tokens = count_tokens(text)
    print(f"  {tokens:4d} tokens | {text[:50]}")

## 3. MessageItem：一条消息的完整表示

原始的 `dict` 格式（`{"role": "user", "content": "..."`）只存了发给 LLM 的数据。

我们需要在此基础上附加一些**内部使用的元数据**——比如这条消息占了多少 token，是不是被截断过。

用 `dataclass` 定义 `MessageItem`，把这些信息放在一起管理。

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class MessageItem:
    """
    一条对话消息，包含发给 LLM 的内容 + 内部追踪数据。
    
    to_dict() 方法只返回 LLM 需要的字段，
    token_count 等内部字段不会发出去。
    """
    role: str                          # "system" / "user" / "assistant" / "tool"
    content: str                       # 消息正文
    token_count: int = 0               # 这条消息占多少 token
    tool_calls: list[dict] | None = None   # assistant 调用工具时，记录调用详情
    tool_call_id: str | None = None    # tool 消息需要对应的调用 ID
    is_truncated: bool = False         # 内容是否被截断过

    def to_dict(self) -> dict:
        """转换为 OpenAI API 需要的格式"""
        msg: dict[str, Any] = {
            "role": self.role,
            "content": self.content,
        }
        # tool_calls 只在 assistant 消息里有（当 AI 决定调用工具时）
        if self.tool_calls:
            msg["tool_calls"] = self.tool_calls
        # tool_call_id 只在 tool 消息里有（返回工具执行结果时）
        if self.tool_call_id:
            msg["tool_call_id"] = self.tool_call_id
        return msg


# 看看几种消息的结构
system_msg = MessageItem(
    role="system",
    content="你是一个代码助手",
    token_count=count_tokens("你是一个代码助手"),
)

user_msg = MessageItem(
    role="user",
    content="帮我写个快速排序",
    token_count=count_tokens("帮我写个快速排序"),
)

print("system 消息发给 LLM 的格式:", system_msg.to_dict())
print(f"  → {system_msg.token_count} tokens")
print()
print("user 消息发给 LLM 的格式:", user_msg.to_dict())
print(f"  → {user_msg.token_count} tokens")

## 4. 系统 Prompt

系统 prompt 是 `role: system` 的消息，放在 `messages` 列表最前面。

它定义了 AI 的**角色、能力、行为规则**。Claude Code 的系统 prompt 据说有几千 token，里面写满了它能做什么、不能做什么、遇到各种情况怎么处理。

我们先写一个简化版，有这几块就够用了：

1. **身份**：它是什么，在做什么
2. **工具使用规则**：什么时候用工具，怎么用
3. **行为准则**：简洁、准确、不乱猜

In [ ]:
def build_system_prompt(working_directory: str = ".") -> str:
    return f"""你是一个 AI 编码助手，帮助用户在本地完成代码任务。

当前工作目录：{working_directory}

## 你的能力
你有一套工具可以调用：读取文件、写入文件、搜索文件、执行网络搜索等。
需要什么信息就去拿，不要凭空猜测。

## 工具使用规则
- 需要看文件内容时，调用 read_file，不要假设文件里写了什么
- 写代码前先了解项目结构，调用 list_directory 或 glob
- 一次只做一件事，完成后再做下一件
- 工具调用失败时，根据错误信息调整参数重试，不要直接放弃

## 回答规则
- 直接回答，不要废话
- 不确定的事情说不确定，不要编造
- 代码用代码块包裹"""


prompt = build_system_prompt("/Users/me/project")
print(prompt)
print()
print(f"系统 prompt 长度：{count_tokens(prompt)} tokens")

## 5. ContextManager 类

把系统 prompt、消息列表、token 统计都放进来。

**设计决定**：
- 系统 prompt 单独存，不放在 `messages` 列表里。调用 `get_messages()` 时再拼上去。这样修改系统 prompt 时不用动消息历史。
- 维护两个 token 计数：`latest_usage`（上一次 LLM 调用的消耗）和 `total_usage`（累计消耗）。后者用来判断要不要压缩。

In [ ]:
from dataclasses import dataclass, field


@dataclass
class UsageStats:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

    def add(self, other: "UsageStats"):
        self.prompt_tokens += other.prompt_tokens
        self.completion_tokens += other.completion_tokens
        self.total_tokens += other.total_tokens


class ContextManager:
    """
    管理 Agent 的对话上下文。
    
    职责：
    - 存储对话历史（MessageItem 列表）
    - 追踪 token 用量
    - 提供 get_messages() 供 LLM 客户端使用
    """

    def __init__(self, system_prompt: str = ""):
        self.system_prompt = system_prompt
        self._messages: list[MessageItem] = []  # 不含 system prompt
        self.latest_usage = UsageStats()         # 上一次 LLM 调用的消耗
        self.total_usage = UsageStats()          # 累计消耗

    # ── 添加消息 ──────────────────────────────────────────────

    def add_user_message(self, content: str) -> MessageItem:
        msg = MessageItem(
            role="user",
            content=content,
            token_count=count_tokens(content),
        )
        self._messages.append(msg)
        return msg

    def add_assistant_message(
        self,
        content: str,
        tool_calls: list[dict] | None = None,
    ) -> MessageItem:
        msg = MessageItem(
            role="assistant",
            content=content,
            token_count=count_tokens(content),
            tool_calls=tool_calls,
        )
        self._messages.append(msg)
        return msg

    def add_tool_result(
        self,
        tool_call_id: str,
        content: str,
    ) -> MessageItem:
        """添加工具执行结果。tool_call_id 对应 assistant 消息里的调用 ID。"""
        msg = MessageItem(
            role="tool",
            content=content,
            token_count=count_tokens(content),
            tool_call_id=tool_call_id,
        )
        self._messages.append(msg)
        return msg

    # ── 获取消息 ──────────────────────────────────────────────

    def get_messages(self) -> list[dict]:
        """返回发给 LLM 的完整消息列表，系统 prompt 放在最前面。"""
        result = []
        if self.system_prompt:
            result.append({"role": "system", "content": self.system_prompt})
        result.extend(msg.to_dict() for msg in self._messages)
        return result

    # ── Token 统计 ────────────────────────────────────────────

    def update_usage(self, prompt_tokens: int, completion_tokens: int):
        """每次 LLM 调用完成后，更新 token 用量。"""
        latest = UsageStats(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=prompt_tokens + completion_tokens,
        )
        self.latest_usage = latest
        self.total_usage.add(latest)

    @property
    def estimated_tokens(self) -> int:
        """估算当前上下文的 token 数（不调用 API，本地计算）。"""
        system_tokens = count_tokens(self.system_prompt) if self.system_prompt else 0
        msg_tokens = sum(m.token_count for m in self._messages)
        return system_tokens + msg_tokens

    def needs_compression(self, context_window: int = 128_000, threshold: float = 0.8) -> bool:
        """判断是否需要压缩历史（超过上下文窗口的 80% 时返回 True）。"""
        return self.estimated_tokens > context_window * threshold

    # ── 工具方法 ──────────────────────────────────────────────

    def clear_messages(self):
        """清空对话历史（保留系统 prompt）。"""
        self._messages.clear()

    def stats(self) -> dict:
        return {
            "message_count": len(self._messages),
            "estimated_tokens": self.estimated_tokens,
            "total_prompt_tokens": self.total_usage.prompt_tokens,
            "total_completion_tokens": self.total_usage.completion_tokens,
            "total_tokens_used": self.total_usage.total_tokens,
        }


print("ContextManager 定义完成")

## 6. 实际使用：配合 LLMClient 做多轮对话

In [ ]:
import sys
sys.path.insert(0, "..")  # 确保能 import src 目录

from src.llm_client import LLMClient, TokenUsage

# 初始化
ctx = ContextManager(system_prompt=build_system_prompt())
llm = LLMClient()

print("初始状态:")
print(f"  系统 prompt: {count_tokens(ctx.system_prompt)} tokens")
print(f"  对话消息:    {len(ctx._messages)} 条")
print(f"  估算总量:    {ctx.estimated_tokens} tokens")

In [ ]:
async def chat_round(user_input: str):
    """进行一轮对话：发消息 → 获取回答 → 更新上下文"""
    
    # 1. 把用户消息加进上下文
    ctx.add_user_message(user_input)
    
    # 2. 发请求（把完整历史发过去）
    text, usage = await llm.chat_completion(ctx.get_messages(), stream=False)
    
    # 3. 把 AI 回答加进上下文
    ctx.add_assistant_message(text)
    
    # 4. 更新 token 统计
    ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)
    
    return text


# 第一轮
reply1 = await chat_round("Python 的 list 和 tuple 有什么区别？")
print("[用户] Python 的 list 和 tuple 有什么区别？")
print(f"[AI]   {reply1[:200]}...")
print()
print("当前统计:", ctx.stats())

In [ ]:
# 第二轮：问「那什么时候用哪个」，模型需要记住上下文才知道「哪个」指的是什么
reply2 = await chat_round("那实际开发中什么时候应该用哪个？")
print("[用户] 那实际开发中什么时候应该用哪个？")
print(f"[AI]   {reply2[:300]}...")
print()
print("当前统计:", ctx.stats())

In [ ]:
# 第三轮：继续追问
reply3 = await chat_round("能给我一个用 tuple 比 list 更合适的代码例子吗？")
print("[用户] 能给我一个用 tuple 比 list 更合适的代码例子吗？")
print(f"[AI]   {reply3[:400]}...")
print()

stats = ctx.stats()
print("最终统计:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print()
print(f"需要压缩吗? {ctx.needs_compression()} (阈值: 128k × 0.8 = 102.4k tokens)")

## 7. 验证 get_messages() 的结构

看看最终发给 LLM 的消息列表长什么样。

In [ ]:
messages = ctx.get_messages()
print(f"共 {len(messages)} 条消息（1 条 system + {len(messages)-1} 条对话）")
print()

for i, msg in enumerate(messages):
    role = msg["role"]
    content_preview = msg["content"][:60].replace("\n", " ")
    print(f"  [{i}] {role:12s} | {content_preview}...")

In [ ]:
# 测试 clear_messages：清空历史，系统 prompt 还在
ctx_test = ContextManager(system_prompt="你是助手")
ctx_test.add_user_message("你好")
ctx_test.add_assistant_message("你好！")

print("清空前:", len(ctx_test.get_messages()), "条消息")
ctx_test.clear_messages()
print("清空后:", len(ctx_test.get_messages()), "条消息")
print("系统 prompt 还在:", ctx_test.get_messages()[0]["role"] == "system")

## 8. 内容截断：超长消息的处理

工具调用的结果有时会很大——比如读了一个 5000 行的文件，全塞进 messages 里很浪费 token。

给 `add_tool_result` 加一个截断：超过阈值就截断，并在末尾标注「已截断」。这样 LLM 知道有内容被省略了，不会误以为拿到了完整内容。

In [ ]:
MAX_TOOL_RESULT_CHARS = 3000  # 工具结果最多保留 3000 字符

def truncate_content(content: str, max_chars: int = MAX_TOOL_RESULT_CHARS) -> tuple[str, bool]:
    """
    截断内容，返回 (处理后的内容, 是否被截断)。
    截断时在末尾加提示，让 LLM 知道内容不完整。
    """
    if len(content) <= max_chars:
        return content, False
    
    truncated = content[:max_chars]
    truncated += f"\n\n[内容已截断，原始长度 {len(content)} 字符，仅显示前 {max_chars} 字符]"
    return truncated, True


# 演示
long_content = "x" * 5000  # 模拟一个很长的工具结果
short, was_truncated = truncate_content(long_content)
print(f"原始: {len(long_content)} 字符")
print(f"截断: {len(short)} 字符，was_truncated={was_truncated}")
print(f"结尾: ...{short[-80:]}")

In [ ]:
# 把截断逻辑加进 ContextManager.add_tool_result
# （给前面定义的类打补丁，实际代码应该直接写在类里）

def add_tool_result_with_truncation(
    self,
    tool_call_id: str,
    content: str,
    max_chars: int = MAX_TOOL_RESULT_CHARS,
) -> MessageItem:
    content_final, is_truncated = truncate_content(content, max_chars)
    msg = MessageItem(
        role="tool",
        content=content_final,
        token_count=count_tokens(content_final),
        tool_call_id=tool_call_id,
        is_truncated=is_truncated,
    )
    self._messages.append(msg)
    return msg

# 绑定到类上（下一章会把截断直接写进类定义里）
import types
ContextManager.add_tool_result = add_tool_result_with_truncation

print("add_tool_result 已更新为带截断版本")

## 9. 小结

| 模块 | 作用 |
|------|------|
| `count_tokens()` | 本地估算 token 数，tiktoken 优先，降级到 len/4 |
| `MessageItem` | 一条消息的完整表示，`to_dict()` 给 LLM，其余字段自用 |
| `build_system_prompt()` | 生成系统 prompt，定义 Agent 身份和行为规则 |
| `ContextManager` | 管理对话历史：添加消息、追踪 token、判断是否需要压缩 |
| `truncate_content()` | 超长工具结果截断，防止单条消息吃掉太多 token |

**关于 `needs_compression()`**：现在只是判断，还没有实现真正的压缩。压缩逻辑在第 08 章：当 token 超过 80% 时，用 LLM 把历史摘要成几百 token，替换掉原来的长历史。

**下一章：Tool Framework**

Context Manager 负责管历史。下一章造工具——Agent 靠工具来操作世界，读文件、写代码、搜索……需要一套统一的接口来定义、注册、调用这些工具。

---
## 附：保存到 src/

In [ ]:
context_manager_code = '''
from dataclasses import dataclass, field
from typing import Any

try:
    import tiktoken
    _encoder = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(_encoder.encode(text))
except ImportError:
    def count_tokens(text: str) -> int:
        return len(text) // 4

MAX_TOOL_RESULT_CHARS = 3000

def truncate_content(content: str, max_chars: int = MAX_TOOL_RESULT_CHARS) -> tuple[str, bool]:
    if len(content) <= max_chars:
        return content, False
    truncated = content[:max_chars]
    truncated += f"\\n\\n[内容已截断，原始长度 {len(content)} 字符，仅显示前 {max_chars} 字符]"
    return truncated, True


@dataclass
class MessageItem:
    role: str
    content: str
    token_count: int = 0
    tool_calls: list[dict] | None = None
    tool_call_id: str | None = None
    is_truncated: bool = False

    def to_dict(self) -> dict:
        msg: dict[str, Any] = {"role": self.role, "content": self.content}
        if self.tool_calls:
            msg["tool_calls"] = self.tool_calls
        if self.tool_call_id:
            msg["tool_call_id"] = self.tool_call_id
        return msg


@dataclass
class UsageStats:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

    def add(self, other: "UsageStats"):
        self.prompt_tokens += other.prompt_tokens
        self.completion_tokens += other.completion_tokens
        self.total_tokens += other.total_tokens


def build_system_prompt(working_directory: str = ".") -> str:
    return f"""你是一个 AI 编码助手，帮助用户在本地完成代码任务。

当前工作目录：{working_directory}

## 你的能力
你有一套工具可以调用：读取文件、写入文件、搜索文件、执行网络搜索等。
需要什么信息就去拿，不要凭空猜测。

## 工具使用规则
- 需要看文件内容时，调用 read_file，不要假设文件里写了什么
- 写代码前先了解项目结构，调用 list_directory 或 glob
- 一次只做一件事，完成后再做下一件
- 工具调用失败时，根据错误信息调整参数重试，不要直接放弃

## 回答规则
- 直接回答，不要废话
- 不确定的事情说不确定，不要编造
- 代码用代码块包裹"""


class ContextManager:
    def __init__(self, system_prompt: str = ""):
        self.system_prompt = system_prompt
        self._messages: list[MessageItem] = []
        self.latest_usage = UsageStats()
        self.total_usage = UsageStats()

    def add_user_message(self, content: str) -> MessageItem:
        msg = MessageItem(role="user", content=content, token_count=count_tokens(content))
        self._messages.append(msg)
        return msg

    def add_assistant_message(self, content: str, tool_calls: list[dict] | None = None) -> MessageItem:
        msg = MessageItem(role="assistant", content=content, token_count=count_tokens(content), tool_calls=tool_calls)
        self._messages.append(msg)
        return msg

    def add_tool_result(self, tool_call_id: str, content: str, max_chars: int = MAX_TOOL_RESULT_CHARS) -> MessageItem:
        content_final, is_truncated = truncate_content(content, max_chars)
        msg = MessageItem(role="tool", content=content_final, token_count=count_tokens(content_final),
                          tool_call_id=tool_call_id, is_truncated=is_truncated)
        self._messages.append(msg)
        return msg

    def get_messages(self) -> list[dict]:
        result = []
        if self.system_prompt:
            result.append({"role": "system", "content": self.system_prompt})
        result.extend(msg.to_dict() for msg in self._messages)
        return result

    def update_usage(self, prompt_tokens: int, completion_tokens: int):
        latest = UsageStats(prompt_tokens=prompt_tokens, completion_tokens=completion_tokens,
                            total_tokens=prompt_tokens + completion_tokens)
        self.latest_usage = latest
        self.total_usage.add(latest)

    @property
    def estimated_tokens(self) -> int:
        system_tokens = count_tokens(self.system_prompt) if self.system_prompt else 0
        return system_tokens + sum(m.token_count for m in self._messages)

    def needs_compression(self, context_window: int = 128_000, threshold: float = 0.8) -> bool:
        return self.estimated_tokens > context_window * threshold

    def clear_messages(self):
        self._messages.clear()

    def stats(self) -> dict:
        return {
            "message_count": len(self._messages),
            "estimated_tokens": self.estimated_tokens,
            "total_prompt_tokens": self.total_usage.prompt_tokens,
            "total_completion_tokens": self.total_usage.completion_tokens,
            "total_tokens_used": self.total_usage.total_tokens,
        }
'''

with open("src/context_manager.py", "w") as f:
    f.write(context_manager_code.strip())

print("已保存到 src/context_manager.py")
print("后续章节用法: from src.context_manager import ContextManager, build_system_prompt")

In [ ]:
await llm.close()